In [ ]:
from sagemaker.pytorch import PyTorch


# Configuration
BUCKET_NAME = "sgp-amc-bucket"
N_JOBS = 5


def launch_job(index):
    hyper = {
            'outputscale': 1.0,
            'se_lengthscale': 1.0,
            'rq_lengthscale': 1.0,
            'rq_alpha': 1.0,
            'per_period_length': 7.0,
            'per_lengthscale': 2.0,
            'noise_variance': 0.020,
            'batch_size': 256,
            'training_iterations': 100,
            'M': 42,
            'lr': 0.009,
            'levels': 2,
            'N_sim': 300, # 10 in 26 minutes
            'index': index}

    # Create the PyTorch estimator with dependencies specified
    estimator = PyTorch(
        entry_point='sgp_training.py',
        source_dir='.',  # Folder containing sgp_training.py and models/
        role="arn:aws:iam::992382529781:role/service-role/AmazonSageMaker-ExecutionRole-20250527T201249",
        instance_count=1,
        instance_type='ml.c5.18xlarge',  # GPU='ml.p3.2xlarge', CPU='ml.m5.large'
        framework_version='1.12',
        py_version='py38',
        hyperparameters=hyper,
        output_path=f's3://{BUCKET_NAME}/models/',
        base_job_name=f'sgp-amc-partition-{index}',
        max_run=31600,  # 9 hours (could not finished for 300 sim; 14400,  # 4 hours
        enable_sagemaker_metrics=True,
        # Add environment variables for GPU optimization
        environment={
            'CUDA_VISIBLE_DEVICES': '0',
            'OMP_NUM_THREADS': '1',  # Prevent CPU thread conflicts
        },
        # Specify dependencies directly instead of requirements.txt
        dependencies=['requirements.txt']
    )

    # Set up input data channels
    train_input = f's3://{BUCKET_NAME}/data/partition{index}/'
    test_input = f's3://{BUCKET_NAME}/data/partition{index}/'

    print(f'\nindex before estimator={index}')
    print(f'train_input=\n {train_input}')

    estimator.fit({
        'train': train_input,
        'test': test_input
    }, wait=False, logs=False)


# Launch training for all partitions in parallel
def train_all_experts(num_partitions: int):
    jobs = []
    for i in range(num_partitions):
        print(f"\nLaunching training for partition {i}...")
        job = launch_job(i)
        jobs.append(job)

    return jobs


# Main execution
if __name__ == "__main__":
    jobs = train_all_experts(num_partitions=N_JOBS)
    print("\nAll training jobs launched")